# Media operations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/media/Document_Media_Operations.ipynb)


## Jump to an API

- [media.load](#mediaload)
- [media.save](#mediasave)
- [media.convert](#mediaconvert)
- [media.copy](#mediacopy)
- [media.save_video](#mediasave_video)
- [media.save_images](#mediasave_images)
- [media.iter_frames](#mediaiter_frames)
- [media.probe](#mediaprobe)
- [media.to_array](#mediato_array)
- [media.to_tensor](#mediato_tensor)
- [media.to_pil](#mediato_pil)
- [media.MediaFrames](#mediamediaframes)
- [media.MediaStream](#mediamediastream)
- [media.LazyImage](#medialazyimage)
- [media.VideoReader](#mediavideoreader)


In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@8e42b64

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
tutorial_workspace = tempfile.TemporaryDirectory()
root = files.path(tutorial_workspace.name)


### media.load

`media.load(source: Union[str, os.PathLike[str]], recursive: bool = False, backend: str = 'pil', verbose: bool = False, *, stream: bool = False, sample_rate: int = 1, max_frames: Optional[int] = None) -> Union[klygo.media.operations.MediaFrames, klygo.media.operations.MediaStream]`

Load image, video, or directory content as lazy frames. Set stream=True to consume frames sequentially.

- Parameters: source is an existing image, video, or directory path; stream, sample_rate, and max_frames control iteration.
- Returns: See the signature and printed result below.
- Errors: FileNotFoundError for missing input; ValueError for invalid backend or stream-only options.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    image = Image.new("RGB", (32, 24), "red")
    source = media.save(root / "source.png", image)
    frames = media.load(source)
    print(len(frames), frames[0].size)
    print(type(media.load(source, stream=True)).__name__)

**Practical variation**

In [ ]:
p = media.save(root / "load_more.png", Image.new("RGB", (8, 8), "blue"))
print(len(list(media.load(p, stream=True))))


**Expected error and recovery**

In [ ]:
try:
    media.load(root / "missing.png")
except FileNotFoundError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.save

`media.save(path: Union[str, pathlib.Path], image: Union[PIL.Image.Image, numpy.ndarray], overwrite: bool = False, verbose: bool = False) -> pathlib.Path`

Write a PIL or NumPy image to a file.

- Parameters: path is the destination; image is a PIL image, array, or LazyImage; overwrite permits replacement.
- Returns: See the signature and printed result below.
- Errors: FileExistsError when the destination exists; TypeError for unsupported image objects.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    destination = media.save(root / "image.png", Image.new("RGB", (32, 24), "red"))
    print(media.probe(destination)["size"])

**Practical variation**

In [ ]:
p = media.save(root / "overwrite.png", Image.new("RGB", (8, 8)))
media.save(p, Image.new("RGB", (8, 8), "blue"), overwrite=True)
print(media.probe(p)["size"])


**Expected error and recovery**

In [ ]:
try:
    p = media.save(root / "exists.png", Image.new("RGB", (8, 8)))
    media.save(p, Image.new("RGB", (8, 8)))
except FileExistsError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.convert

`media.convert(source: Union[str, pathlib.Path], target: Union[str, pathlib.Path], codec: Optional[str] = None, crf: int = 23, fps: Optional[float] = None, gpu: Optional[bool] = None, preset: Optional[str] = None, overwrite: bool = False, verbose: bool = False) -> pathlib.Path`

Convert one image format to another, or transcode a video.

- Parameters: source and target determine media formats; codec, crf, fps, gpu, and preset tune video conversion.
- Returns: See the signature and printed result below.
- Errors: FileNotFoundError for missing source; ValueError for incompatible media kinds.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    image = Image.new("RGB", (32, 24), "red")
    source = media.save(root / "source.png", image)
    target = media.convert(source, root / "converted.jpg")
    print(media.probe(target)["format"])

**Practical variation**

In [ ]:
p = media.save(root / "convert_more.png", Image.new("RGB", (8, 8)))
print(media.convert(p, root / "convert_more.webp"))


**Expected error and recovery**

In [ ]:
try:
    p = media.save(root / "wrong_kind.png", Image.new("RGB", (8, 8)))
    media.convert(p, root / "wrong_kind.mp4")
except ValueError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.copy

`media.copy(source: Union[str, pathlib.Path], target: Union[str, pathlib.Path], overwrite: bool = False) -> pathlib.Path`

Copy a media file or directory.

- Parameters: source is an existing path; target is the new path; overwrite controls replacement.
- Returns: See the signature and printed result below.
- Errors: FileNotFoundError for missing source; FileExistsError if target exists without overwrite.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    image = Image.new("RGB", (32, 24), "red")
    source = media.save(root / "source.png", image)
    duplicate = media.copy(source, root / "copy.png")
    print(media.probe(duplicate)["size"])

**Practical variation**

In [ ]:
p = media.save(root / "copy_more.png", Image.new("RGB", (8, 8)))
print(media.copy(p, root / "copy_more_2.png"))


**Expected error and recovery**

In [ ]:
try:
    media.copy(root / "missing.png", root / "target.png")
except FileNotFoundError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.save_video

`media.save_video(output_path: Union[str, pathlib.Path], frames: Iterable[Union[PIL.Image.Image, numpy.ndarray]], fps: float = 30.0, fourcc: str = 'mp4v', overwrite: bool = False, verbose: bool = False) -> pathlib.Path`

Encode an iterable of equally sized frames as a video.

- Parameters: output_path is a video path; frames is an iterable; fps and fourcc control encoding.
- Returns: See the signature and printed result below.
- Errors: ValueError for empty frames; RuntimeError if the video writer cannot open.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    frames = [Image.new("RGB", (32, 24), color) for color in ("red", "blue")]
    video = media.save_video(root / "clip.mp4", frames, fps=5)
    print(media.probe(video)["frame_count"])

**Practical variation**

In [ ]:
clip = media.save_video(root / "short.avi", [Image.new("RGB", (16, 16), "blue")], fps=2, fourcc="MJPG")
print(media.probe(clip)["type"])


**Expected error and recovery**

In [ ]:
try:
    media.save_video(root / "empty.mp4", [])
except ValueError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.save_images

`media.save_images(output_dir: Union[str, pathlib.Path], images: Iterable[Union[PIL.Image.Image, numpy.ndarray]], prefix: str = 'frame', extension: str = '.jpg', overwrite: bool = False, verbose: bool = False) -> List[pathlib.Path]`

Write a sequence of numbered image files.

- Parameters: output_dir receives generated filenames; prefix and extension customize them; overwrite controls collisions.
- Returns: See the signature and printed result below.
- Errors: FileExistsError when a generated name exists without overwrite.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    frames = [Image.new("RGB", (32, 24), color) for color in ("red", "blue")]
    paths = media.save_images(root / "frames", frames, extension=".png")
    print([files.name(path) for path in paths])

**Practical variation**

In [ ]:
paths = media.save_images(root / "batch_more", [Image.new("RGB", (8, 8))], prefix="snapshot", extension="png")
print(files.name(paths[0]))


**Expected error and recovery**

In [ ]:
try:
    batch = root / "collision"
    media.save_images(batch, [Image.new("RGB", (8, 8))])
    media.save_images(batch, [Image.new("RGB", (8, 8))])
except FileExistsError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.iter_frames

`media.iter_frames(source: Union[str, pathlib.Path], sample_rate: int = 1, recursive: bool = False, backend: str = 'pil', verbose: bool = False) -> Generator[Union[PIL.Image.Image, numpy.ndarray], NoneType, NoneType]`

Yield sampled frames from a video or image directory.

- Parameters: source is an image directory or video; sample_rate takes every nth frame; backend selects PIL or OpenCV.
- Returns: See the signature and printed result below.
- Errors: ValueError for sample_rate below one; FileNotFoundError for missing input.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    image = Image.new("RGB", (32, 24), "red")
    source = media.save(root / "source.png", image)
    print([frame.size for frame in media.iter_frames(root, sample_rate=1)])

**Practical variation**

In [ ]:
media.save(root / "iter_more.png", Image.new("RGB", (8, 8)))
print(len(list(media.iter_frames(root, sample_rate=2))))


**Expected error and recovery**

In [ ]:
try:
    list(media.iter_frames(root, sample_rate=0))
except ValueError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.probe

`media.probe(path: Union[str, pathlib.Path]) -> Dict[str, Any]`

Inspect image or video metadata without loading a frame collection.

- Parameters: path is an existing media file.
- Returns: See the signature and printed result below.
- Errors: FileNotFoundError for missing file; ValueError for a directory.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    image = Image.new("RGB", (32, 24), "red")
    source = media.save(root / "source.png", image)
    print(media.probe(source)["type"], media.probe(source)["size"])

**Practical variation**

In [ ]:
clip = media.save_video(root / "probe_more.mp4", [Image.new("RGB", (16, 16))], fps=3)
print(media.probe(clip)["type"])


**Expected error and recovery**

In [ ]:
try:
    media.probe(root)
except ValueError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.to_array

`media.to_array(image: Any) -> numpy.ndarray`

Convert PIL, NumPy, LazyImage, or Torch image data into a NumPy array.

- Parameters: image is an accepted image representation.
- Returns: See the signature and printed result below.
- Errors: TypeError for unsupported input.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    array = media.to_array(Image.new("RGB", (32, 24), "red"))
    print(array.shape, array.dtype)

**Practical variation**

In [ ]:
array = media.to_array(Image.new("RGB", (8, 8)))
array[0, 0] = (255, 0, 0)
print(array[0, 0].tolist())


**Expected error and recovery**

In [ ]:
try:
    media.to_array(object())
except TypeError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.to_tensor

`media.to_tensor(image: Any, normalize: bool = True) -> Any`

Convert image data to a CHW PyTorch tensor.

- Parameters: image is an accepted image representation; normalize scales uint8 pixels to float values in 0–1.
- Returns: See the signature and printed result below.
- Errors: ImportError if PyTorch is unavailable; TypeError for unsupported input.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    tensor = media.to_tensor(Image.new("RGB", (32, 24), "red"), normalize=True)
    print(tuple(tensor.shape), str(tensor.dtype))

**Practical variation**

In [ ]:
tensor = media.to_tensor(Image.new("RGB", (8, 8)), normalize=False)
print(tuple(tensor.shape), str(tensor.dtype))


**Expected error and recovery**

In [ ]:
try:
    media.to_tensor(object())
except TypeError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.to_pil

`media.to_pil(image: Any) -> PIL.Image.Image`

Convert an array, tensor, or lazy image to a PIL image.

- Parameters: image is a PIL, NumPy, Torch, or LazyImage image.
- Returns: See the signature and printed result below.
- Errors: TypeError for unsupported data; ValueError for unsupported array shape.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    import numpy as np
    image = media.to_pil(np.zeros((24, 32, 3), dtype=np.uint8))
    print(image.size, image.mode)

**Practical variation**

In [ ]:
import numpy as np
print(media.to_pil(np.zeros((8, 8), dtype=np.uint8)).mode)


**Expected error and recovery**

In [ ]:
try:
    import numpy as np
    media.to_pil(np.zeros((8, 8, 2), dtype=np.uint8))
except ValueError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.MediaFrames

`media.MediaFrames(items: Optional[Iterable] = None, *, fps: float = 30.0, source_path: Union[str, pathlib.Path, NoneType] = None, width: Optional[int] = None, height: Optional[int] = None, source_type: str = 'video', reader: Optional[klygo.media.operations.VideoReader] = None) -> None`

Hold indexable lazy image frames and source metadata.

- Parameters: Construct from frame proxies; optional fps, source path, dimensions, and reader preserve media context.
- Returns: See the signature and printed result below.
- Errors: IndexError for an index outside the collection.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    image = Image.new("RGB", (32, 24), "red")
    source = media.save(root / "source.png", image)
    frames = media.load(root)
    print(isinstance(frames, media.MediaFrames), frames.total_frames, frames.loaded_count)

**Practical variation**

In [ ]:
p = media.save(root / "frames_more.png", Image.new("RGB", (8, 8)))
frames = media.load(p)
print(type(frames[:1]).__name__, frames.loaded_count)


**Expected error and recovery**

In [ ]:
try:
    media.MediaFrames([])[0]
except IndexError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.MediaStream

`media.MediaStream(source: Union[str, os.PathLike[str]], *, sample_rate: int = 1, max_frames: Optional[int] = None, recursive: bool = False, backend: str = 'pil', verbose: bool = False) -> None`

Iterate frames from a source without retaining the whole decoded video.

- Parameters: Construct from source plus sampling, recursion, backend, and frame-limit options.
- Returns: See the signature and printed result below.
- Errors: FileNotFoundError for missing source; ValueError for invalid sampling.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    image = Image.new("RGB", (32, 24), "red")
    source = media.save(root / "source.png", image)
    with media.load(root, stream=True) as stream:
        print(isinstance(stream, media.MediaStream), [frame.size for frame in stream])

**Practical variation**

In [ ]:
p = media.save(root / "stream_more.png", Image.new("RGB", (8, 8)))
with media.load(p, stream=True, max_frames=1) as stream:
    print(next(iter(stream)).size)


**Expected error and recovery**

In [ ]:
try:
    media.MediaStream(root / "missing")
except FileNotFoundError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.LazyImage

`media.LazyImage(path: Union[str, pathlib.Path, ForwardRef('LazyImage')], backend: str = 'pil', crop_box: Optional[Tuple[int, int, int, int]] = None, frame_index: Optional[int] = None, reader: Optional[klygo.media.operations.VideoReader] = None) -> None`

Represent an image by path or frame reference and decode pixels on demand.

- Parameters: Construct from a source image path or image object; optional backend and frame context tune loading.
- Returns: See the signature and printed result below.
- Errors: FileNotFoundError for missing source when pixels are requested.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    image = Image.new("RGB", (32, 24), "red")
    source = media.save(root / "source.png", image)
    lazy = media.LazyImage(source)
    print(lazy.size, lazy.is_loaded)
    print(lazy.to_array().shape)

**Practical variation**

In [ ]:
p = media.save(root / "lazy_more.png", Image.new("RGB", (8, 8)))
lazy = media.LazyImage(p)
print(lazy.is_loaded, lazy.to_pil(cache=True).size)


**Expected error and recovery**

In [ ]:
try:
    media.LazyImage(root / "missing.png").to_pil()
except FileNotFoundError as error:
    print(type(error).__name__, "Correct the input and retry")


### media.VideoReader

`media.VideoReader(path: Union[str, pathlib.Path]) -> None`

Reuse one OpenCV capture for sequential and random video-frame access.

- Parameters: path is an existing video file.
- Returns: See the signature and printed result below.
- Errors: FileNotFoundError for missing path or unreadable video.

**Basic use**

In [ ]:
import tempfile
from PIL import Image
from klygo import media, files
with tempfile.TemporaryDirectory() as temporary:
    root = files.path(temporary)
    video = media.save_video(root / "clip.mp4", [Image.new("RGB", (32, 24), "red")], fps=5)
    reader = media.VideoReader(video)
    try:
        print(reader.read_frame(0).shape)
    finally:
        reader.close()

**Practical variation**

In [ ]:
p = media.save_video(root / "reader_more.mp4", [Image.new("RGB", (16, 16))], fps=2)
reader = media.VideoReader(p)
try:
    print(reader.read_frame(0).shape)
finally:
    reader.close()


**Expected error and recovery**

In [ ]:
try:
    media.VideoReader(root / "missing.mp4")
except FileNotFoundError as error:
    print(type(error).__name__, "Correct the input and retry")


## Put it together

In [ ]:
image = Image.new("RGB", (48, 32), "green")
source = media.save(root / "workflow.png", image)
converted = media.convert(source, root / "workflow.jpg")
print(media.probe(converted)["format"], media.load(converted)[0].size)


## Remember

Use `media.load` for lazy indexed frames and `media.iter_frames` or `MediaStream` for sequential work. Output paths need `overwrite=True` for replacement. Return to the [Media overview](Document_Media_Overview.ipynb), [Files overview](../files/Document_Files_Overview.ipynb), or [Visual overview](../visual/Document_Visual_Overview.ipynb).

In [ ]:
tutorial_workspace.cleanup()